# 3장 예제 — 3.6 시계열 데이터 다루기 기초

- **교재 출처**: manuscript/30_ch3_전처리.md
- **실행 방법**: example 폴더에서  python "3장/<이 파일>" (교재 코드의 data/ 상대경로를 그대로 쓰기 위해 작업 디렉터리는 example)
- **선행 필요**: 이 절의 코드는 앞 절에서 만든 변수를 이어 쓴다. 단독 실행이 필요하면 같은 장의 _장전체_실행본.py 를 쓰거나 아래 절을 먼저 실행하라 — 03_3-3_이상치_처리.py, 04_3-4_중복·형변환·단위_정리.py, 05_3-5_스케일링.py
- **실행 상태**: 단독 실행 확인됨(이 파일만 돌려도 끝까지 실행된다).

In [ ]:
# [환경 설정] data/ 경로 자동 감지
import os
if not os.path.exists('data') and os.path.exists('../data'):
    os.chdir('..')

In [ ]:
# --- 실행 준비 (교재에서는 앞 절에서 이미 실행한 코드) --------------------
import pandas as pd
import numpy as np

### 합성 센서 로그 생성과 datetime 파싱

In [ ]:
rng = np.random.default_rng(42)
idx = pd.date_range('2026-03-02 08:00', periods=600, freq='10s')
sensor = pd.DataFrame({
    'timestamp': idx,
    '건조로_온도': 120 + 3*np.sin(np.arange(600)/50) + rng.normal(0, 0.8, 600)
})
sensor.loc[300:305, '건조로_온도'] += 15   # 순간 스파이크 주입
print(sensor.head())
print("전체 구간:", sensor['timestamp'].min(), "~", sensor['timestamp'].max())

# 교재 실행 결과 ------------------------------------------------------
#               timestamp      건조로_온도
#   0 2026-03-02 08:00:00  120.243774
#   1 2026-03-02 08:00:10  119.228009
#   2 2026-03-02 08:00:20  120.720329
#   3 2026-03-02 08:00:30  120.932344
#   4 2026-03-02 08:00:40  118.678916
#   전체 구간: 2026-03-02 08:00:00 ~ 2026-03-02 09:39:50

In [ ]:
sensor['timestamp'] = pd.to_datetime(sensor['timestamp'])
s = sensor.set_index('timestamp')      # 시각을 인덱스로

### 리샘플링 — 시간 해상도 바꾸기

In [ ]:
res = s.resample('1min').mean()
print(res.head())
print("리샘플링 후 행 수:", len(res))

# 교재 실행 결과 ------------------------------------------------------
#                            건조로_온도
#   timestamp                      
#   2026-03-02 08:00:00  119.843521
#   2026-03-02 08:01:00  120.587110
#   2026-03-02 08:02:00  120.885537
#   2026-03-02 08:03:00  121.332563
#   2026-03-02 08:04:00  121.643828
#   리샘플링 후 행 수: 100

### 이동평균 — 잡음 속에서 추세 읽기

In [ ]:
s['이동평균_1min'] = s['건조로_온도'].rolling('60s').mean()
print(s.iloc[298:308].round(2))

# 교재 실행 결과 ------------------------------------------------------
#                        건조로_온도  이동평균_1min
#   timestamp                             
#   2026-03-02 08:49:40  117.64     118.31
#   2026-03-02 08:49:50  119.37     118.75
#   2026-03-02 08:50:00  135.54     121.57
#   2026-03-02 08:50:10  132.99     123.90
#   2026-03-02 08:50:20  134.97     126.53
#   2026-03-02 08:50:30  134.07     129.10
#   2026-03-02 08:50:40  134.35     131.88
#   2026-03-02 08:50:50  133.61     134.26
#   2026-03-02 08:51:00  119.25     131.54
#   2026-03-02 08:51:10  120.61     129.48

### 시계열 동기화 개요 — 로트와 센서 로그 잇기

In [ ]:
lot_log = pd.DataFrame({
    'timestamp': pd.to_datetime(['2026-03-02 08:07:23',
                                 '2026-03-02 08:31:11',
                                 '2026-03-02 08:55:40']),
    'Lot_ID': ['L25101', 'L25102', 'L25103']
})
sync = pd.merge_asof(lot_log, sensor.sort_values('timestamp'),
                     on='timestamp', direction='nearest')
print(sync)

# 교재 실행 결과 ------------------------------------------------------
#               timestamp  Lot_ID      건조로_온도
#   0 2026-03-02 08:07:23  L25101  122.405565
#   1 2026-03-02 08:31:11  L25102  117.330443
#   2 2026-03-02 08:55:40  L25103  121.185673